# MethodThinker 训练 Notebook

> 全量数据训练 (13,340样本) | QLoRA模式 | T4 GPU优化

## 训练配置
- **模型**: Qwen/Qwen2.5-Math-1.5B
- **数据**: 13,340样本 (GSM8K + Omni-MATH + 原始数据)
- **模式**: QLoRA (4-bit量化 + LoRA)
- **预计时间**: 30-60分钟 (T4 GPU)

## 1. 环境准备

In [ ]:
# 检查GPU
import torch
print("="*50)
print("GPU信息")
print("="*50)
if torch.cuda.is_available():
    print(f"✓ GPU型号: {torch.cuda.get_device_name(0)}")
    print(f"✓ 显存大小: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"✓ CUDA版本: {torch.version.cuda}")
    print(f"✓ 支持BF16: {torch.cuda.is_bf16_supported()}")
else:
    print("✗ 未检测到GPU!")
    print("请启用GPU: 运行时 → 更改运行时类型 → T4 GPU")

In [ ]:
# 克隆项目
!git clone https://github.com/alasong/method-thinker.git
%cd method-thinker
print("\n✓ 项目已克隆")

In [ ]:
# 安装依赖
print("安装依赖...")
!pip install -q transformers>=4.40.0 accelerate>=0.25.0 peft>=0.7.0 trl>=0.7.0 datasets>=2.14.0 bitsandbytes>=0.41.0
print("✓ 依赖安装完成")

In [ ]:
# 验证安装
import transformers, peft, trl
print(f"✓ transformers: {transformers.__version__}")
print(f"✓ peft: {peft.__version__}")
print(f"✓ trl: {trl.__version__}")

## 2. 数据准备

In [ ]:
# 检查训练数据
import json
import os

data_path = "data/train_data/all_merged.json"

if os.path.exists(data_path):
    with open(data_path, 'r') as f:
        data = json.load(f)
    print(f"✓ 训练数据: {len(data)} 样本")
    print(f"✓ 数据来源: GSM8K + Omni-MATH + 原始数据")
    
    # 查看样本结构
    print(f"\n样本字段: {list(data[0].keys())}")
else:
    print("✗ 训练数据不存在，请先运行数据下载")

## 3. 开始训练

In [ ]:
# 训练命令
!python scripts/train_sft.py \
    --train-data data/train_data/all_merged.json \
    --use-lora \
    --epochs 3 \
    --batch-size 4 \
    --max-length 2048 \
    --output-dir outputs/models/v1

## 4. 保存结果

In [ ]:
# 挂载Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 保存模型到Drive
import shutil
import os

save_dir = '/content/drive/MyDrive/method-thinker-models/v1'
os.makedirs(save_dir, exist_ok=True)

# 复制模型
if os.path.exists('outputs/models/v1'):
    shutil.copytree('outputs/models/v1', save_dir, dirs_exist_ok=True)
    print(f"✓ 模型已保存到: {save_dir}")
    
    # 列出保存的文件
    print("\n保存的文件:")
    for f in os.listdir(save_dir):
        fpath = os.path.join(save_dir, f)
        if os.path.isfile(fpath):
            size = os.path.getsize(fpath) / 1024 / 1024
            print(f"  {f}: {size:.1f} MB")
else:
    print("✗ 模型目录不存在")

## 5. 验证模型

In [ ]:
# 测试模型推理
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "outputs/models/v1"

print("加载模型...")
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    torch_dtype=torch.float16
)

# 测试问题
test_problem = """解方程: 2x + 5 = 13
请给出解题步骤。"""

print(f"\n测试问题: {test_problem}")
print("\n模型回答:")

inputs = tokenizer(test_problem, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.7)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## 6. 清理（可选）

In [ ]:
# 取消挂载Drive
drive.flush_and_unmount()
print("✓ Google Drive已卸载")